In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import pickle
import json
import copy

In [ ]:
# for plotting

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
rundirs = [
    'bomex_25m_r20251009',
    'bomex_25m_ehe18_r20251009',
    'bomex_25m_ehe22_r20251107',
    'bomex_25m_ehe21_r20251107',
]
casenames = [
    'CTL',
    'EHEall',
    'EHElower',
    'EHEupper',
]
casecolors = [ 
    'black',
    'green',
    'red',
    'blue',
]
stats = [] 
for rundir in rundirs:
    with open(f'{rundir}/pkl/csd_stats.pkl', 'rb') as f:
        stats.append(pickle.load(f))

In [ ]:
nx, ny, nz, nt = 512, 512, 120, 241
# dts = 0.5 # minute
dts = 30 # seconds
dx = 25 # m
dy = 25 # m
dz = 25 # m
grid_vol = dx*1.0e-3*dy*1.0e-3*dz*1.0e-3 # km**3
z = np.arange(dz/2, 3000., dz)
t = np.arange(0, 241)*dts # minutes
ti = np.arange(-0.5, 241., 1.)*dts
zi = np.arange(0., 3001., dz)

In [ ]:
minmf = 0
maxmf = np.max([np.max(s[0]) for s in stats]) + 10.0
maxmf = np.log10(maxmf)
print(minmf, maxmf)

In [ ]:
def calc_cloud_lifetime(stats, casename, bins, dts):

    clipped_mf = stats[0]
    cloud_times = stats[3]
    mctl_c = stats[5]
    nbins = len(bins) - 1
    colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))

    n, _ = np.histogram(np.log10(clipped_mf), bins=bins)
    sum_t, _ = np.histogram(
        np.log10(clipped_mf),
        bins=bins,
        weights=cloud_times * dts)
    ave_t = np.asarray(sum_t) / np.asarray(n, dtype=float)
    bcs = (bins[1:] + bins[:-1]) * 0.5
    bin_ids = np.digitize(np.log10(clipped_mf), bins) - 1

    fig = plt.figure(figsize=(16, 8))
    ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
    for i, c in enumerate(colors):
        ax.scatter(
            np.log10(clipped_mf[bin_ids == i]),
            np.log10(cloud_times[bin_ids == i] * dts),
            s=5,
            color=c,
        )
        ax.plot(bcs[i], np.log10(ave_t[i]), marker="s", markersize=20, color='black')
    ax.set_xlabel(
        r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
    )
    ax.set_ylabel("Cloud Lifetime (secs, log-10-scale)")
    ax.set_title(f"{casename}", fontsize=20)
    plt.show()

    sum_zt, _ = np.histogram(
        np.log10(clipped_mf), bins=bins, weights=z[mctl_c]
    )
    ave_zt = np.asarray(sum_zt) / np.asarray(n, dtype=float)
    fig = plt.figure(figsize=(16, 8))
    ax = fig.add_axes((0.05, 0.05, 0.9, 0.9))
    colors = plt.cm.rainbow_r(np.linspace(0, 1, nbins))
    for i, c in enumerate(colors):
        ax.scatter(np.log10(clipped_mf[bin_ids == i]), z[mctl_c[bin_ids == i]], s=5, color=c)
        ax.plot(bcs[i], ave_zt[i], marker="s", markersize=20, color='black')
    ax.set_xlabel(
        r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20
    )
    ax.set_ylabel("Cloud Top Height (m)")
    ax.set_title(f"{casename}", fontsize=20)
    plt.show()

    return ave_t, ave_zt

In [ ]:
bins = np.linspace(minmf, maxmf, 11)
bcs = (bins[1:] + bins[:-1]) * 0.5

In [ ]:
ave_t_list = []
ave_zt_list = []
for i in range(len(casenames)):
    ave_t, ave_zt = calc_cloud_lifetime(stats[i], casenames[i], bins, dts)
    ave_t_list.append(ave_t)
    ave_zt_list.append(ave_zt)

In [ ]:
bcs = (bins[1:] + bins[:-1]) * 0.5

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Plot ave_t for all cases
for i in range(len(casenames)):
    ax1.plot(bcs, ave_t_list[i], marker='o', markersize=10, 
             label=casenames[i], color=casecolors[i], linewidth=2)
ax1.set_xlabel(r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20)
ax1.set_ylabel("Cloud Lifetime (secs)", fontsize=20)
# ax1.set_ylabel("Cloud Lifetime (secs, log-10-scale)", fontsize=20)
ax1.legend(fontsize=16)
ax1.set_title("Cloud Lifetime vs Mass Flux", fontsize=20)
ax1.grid(True, alpha=0.3)

# Plot ave_zt for all cases
for i in range(len(casenames)):
    ax2.plot(bcs, ave_zt_list[i], marker='o', markersize=10, 
             label=casenames[i], color=casecolors[i], linewidth=2)
ax2.set_xlabel(r"Lifetime-Mean Plume-Area Cloud-Base Mass-Flux ($kg/s$, log-10-scale)", fontsize=20)
ax2.set_ylabel("Cloud Top Height (m)", fontsize=20)
ax2.legend(fontsize=16)
ax2.set_title("Cloud Top Height vs Mass Flux", fontsize=20)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()